In [ ]:
from dotenv import load_dotenv

from langchain import hub
from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import chain
from langchain_teddynote.messages import stream_response
from langchain_teddynote.callbacks import StreamingCallback
from langchain_core.output_parsers import StrOutputParser

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-summary-mapreduce")

# Map-Reduce

1. Map: 문서를 우선 작은 chunk로 나눈 뒤, 각 chunk를 병렬로 요약
2. Reduce: 각 chunk의 요약을 하나의 최종 요약으로 통합합

대규모 문서 처리 시 특히 유용하고, 언어 모델의 토큰 제한을 우회하게 해줌.

## Map

In [ ]:
FILE_PATH = "C:/Users/grego/experiment/AI4CEO/data/SPRI_AI_Brief_2023년12월호.pdf"

loader_pdf = PyPDFLoader(FILE_PATH)
docs_pdf = loader_pdf.load()
docs_pdf = docs_pdf[3:8]

print(f"총 페이지수: {len(docs_pdf)}")

In [ ]:
llm_2 = ChatOpenAI(
    temperature=0, 
    model_name="gpt-4o-mini"
)

In [ ]:
map_prompt = hub.pull("teddynote/map-prompt")  # map prompt 다운로드

map_prompt.pretty_print()

In [ ]:
# map chain 생성
map_chain = map_prompt | llm | StrOutputParser()

In [ ]:
# 문서에 대한 주요내용 추출. 각 문서에 대한 요약본 생성.
doc_pdf_summaries = map_chain.batch(docs_pdf)

In [ ]:
print(len(doc_summaries))  # 요약된 문서의 수
print(doc_summaries[0])  # 일부 문서의 요약

## Reduce

In [ ]:
reduce_prompt = hub.pull("teddynote/reduce-prompt")

reduce_prompt.pretty_print()

In [ ]:
# reduce chain 생성
reduce_chain = reduce_prompt | llm | StrOutputParser()

In [ ]:
answer = reduce_chain.stream(
    {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
)

In [ ]:
stream_response(answer)

In [ ]:
@chain
def map_reduce_chain(docs):
    map_llm = ChatOpenAI(
        temperature=0,
        model_name="gpt-4o-mini",
    )

    # map prompt 다운로드
    map_prompt = hub.pull("teddynote/map-prompt")

    # map chain 생성
    map_chain = map_prompt | map_llm | StrOutputParser()

    # 첫 번째 프롬프트, ChatOpenAI, 문자열 출력 파서를 연결하여 체인을 생성합니다.
    doc_summaries = map_chain.batch(docs)

    # reduce prompt 다운로드
    reduce_prompt = hub.pull("teddynote/reduce-prompt")
    reduce_llm = ChatOpenAI(
        model_name="gpt-4o",
        temperature=0,
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    reduce_chain = reduce_prompt | reduce_llm | StrOutputParser()

    return reduce_chain.invoke(
        {"doc_summaries": "\n".join(doc_summaries), "language": "Korean"}
    )

In [ ]:
map_reduce_chain.invoke(docs)